# Anomaly detection

Stage 4 left us with two years of daily alerts for every territory. This notebook uses that history to narrow things down to a short list of territories where forest loss is unusually high for that territory right now.

The main decisions are: how to group the data over time, what to use as the baseline, what to do with territories that don't have enough history, and how to rank the final results.

In [1]:
import pandas as pd
import geopandas as gpd

alerts = pd.read_csv("../data/processed/alerts_daily.csv", parse_dates=["date"])
territories = gpd.read_parquet("../data/processed/territories.parquet")

print(f"{len(alerts):,} alert rows, {len(territories):,} territories")

647,754 alert rows, 3,866 territories


The evaluation window is seven days and gets recalculated every day. The baseline, however, is built using weeks. At daily resolution, about half the territories are mostly zeros, so there isn't much of a distribution to compare against.

A week with no alerts is a real zero, not missing data. If those weeks aren't filled in, each territory's history would only include weeks where it lost forest, which would push the baseline higher than it should be.

In [2]:
last_day = alerts["date"].max()
window = alerts[alerts["date"] > last_day - pd.Timedelta(days=7)]

recent = window.groupby("territory_id")["area_ha"].sum().rename("lost_ha")

ranking = (
    territories[["territory_id", "name", "country", "area_ha"]]
    .merge(recent, on="territory_id", how="left")
    .fillna({"lost_ha": 0})
)
ranking["lost_per_1000ha"] = ranking["lost_ha"] / ranking["area_ha"] * 1000

top = ranking.nlargest(20, "lost_per_1000ha")
print(f"window: {(last_day - pd.Timedelta(days=7)).date()} to {last_day.date()}\n")
print(top[["name", "country", "area_ha", "lost_ha", "lost_per_1000ha"]].round(3).to_string(index=False))

window: 2026-08-05 to 2026-08-12

                          name country   area_ha  lost_ha  lost_per_1000ha
               Ñukanchi Llacta Ecuador   442.069    2.646            5.986
                       Remanso    Perú  1647.255    7.567            4.593
                        Ñeñeña Ecuador   147.564    0.677            4.588
                     Morroyacu    Perú 14324.122   61.660            4.305
                        Yarina    Perú  3649.511   14.949            4.096
                  Puerto Nuevo    Perú  1202.211    4.472            3.720
                   17 De Abril Ecuador   433.192    1.489            3.438
               Territorio Iwia Ecuador   359.577    1.206            3.355
                  Santa Isabel    Perú  1693.399    4.956            2.927
               Puerto Porvenir    Perú  2426.692    7.087            2.921
                   San Antonio    Perú  2015.440    5.564            2.761
                      Arazaire    Perú  1331.025    3.297         

In [3]:
for t in [1, 2, 5, 10, 25, 50]:
    print(f"at least {t:>3} ha: {(ranking['lost_ha'] >= t).sum():>4} territories")

at least   1 ha:  644 territories
at least   2 ha:  399 territories
at least   5 ha:  182 territories
at least  10 ha:   93 territories
at least  25 ha:   41 territories
at least  50 ha:   24 territories


In [4]:
MIN_LOST_HA = 5

eligible = ranking[ranking["lost_ha"] >= MIN_LOST_HA]
top = eligible.nlargest(20, "lost_per_1000ha")

print(f"{len(eligible)} territories above {MIN_LOST_HA} ha this week\n")
print(top[["name", "country", "area_ha", "lost_ha", "lost_per_1000ha"]].round(2).to_string(index=False))

182 territories above 5 ha this week

                             name country  area_ha  lost_ha  lost_per_1000ha
                          Remanso    Perú  1647.26     7.57             4.59
                        Morroyacu    Perú 14324.12    61.66             4.30
                           Yarina    Perú  3649.51    14.95             4.10
                  Puerto Porvenir    Perú  2426.69     7.09             2.92
                      San Antonio    Perú  2015.44     5.56             2.76
                     Santa Martha    Perú  6652.60    12.90             1.94
                            Sebai  Guyana  8722.24    16.55             1.90
                   Nuevo Mayoruna    Perú  2721.70     5.05             1.85
             Territorio Yamanunka Ecuador  9119.95    16.62             1.82
                          Umutina  Brasil 27721.82    50.27             1.81
                   Puerto Amistad    Perú  4315.46     7.61             1.76
       Sinchi Roca - Ampliación I    P

In [5]:
alerts["week"] = alerts["date"].dt.to_period("W").dt.start_time

weekly = (
    alerts.groupby(["territory_id", "week"])["area_ha"]
    .sum()
    .reset_index()
)

full_index = pd.MultiIndex.from_product(
    [territories["territory_id"].unique(),
     pd.date_range(weekly["week"].min(), weekly["week"].max(), freq="W-MON")],
    names=["territory_id", "week"],
)
weekly = weekly.set_index(["territory_id", "week"]).reindex(full_index, fill_value=0).reset_index()

weekly = weekly.merge(
    territories[["territory_id", "area_ha"]].rename(columns={"area_ha": "territory_ha"}),
    on="territory_id",
)
weekly["lost_per_1000ha"] = weekly["area_ha"] / weekly["territory_ha"] * 1000

print(f"{len(weekly):,} territory-weeks")
print(f"{weekly['week'].nunique()} weeks")
print(f"{weekly['territory_id'].nunique():,} territories")

413,662 territory-weeks
107 weeks
3,866 territories


Yanomami has alerts in almost every week across the two years. San Pedro, which is around the median in size, has no alerts in about a third of its weeks.

In [6]:
def show(tid, name):
    s = weekly[weekly["territory_id"] == tid].set_index("week")["lost_per_1000ha"]
    print(f"\n{name} (id {tid})")
    print(f"  weeks with zero loss: {(s == 0).sum()} of {len(s)}")
    print(s.describe().round(4).to_string())

show(4016, "Yanomami")
show(40309, "San Pedro")


Yanomami (id 4016)
  weeks with zero loss: 0 of 107
count    107.0000
mean       0.0151
std        0.0253
min        0.0002
25%        0.0034
50%        0.0073
75%        0.0122
max        0.1544

San Pedro (id 40309)
  weeks with zero loss: 36 of 107
count    107.0000
mean       0.0356
std        0.0557
min        0.0000
25%        0.0000
50%        0.0117
75%        0.0395
max        0.2811


If most territories were mostly zeros, a percentile-based method wouldn't work, and we'd need two separate criteria.

In [7]:
zero_share = (
    weekly.assign(is_zero=weekly["lost_per_1000ha"] == 0)
    .groupby("territory_id")["is_zero"]
    .mean()
)

print(zero_share.describe().round(3).to_string())
print()
for t in [0.25, 0.5, 0.7, 0.8, 0.9, 0.95]:
    n = (zero_share > t).sum()
    print(f"more than {t:.0%} zero weeks: {n:>4} territories ({n/len(zero_share):.1%})")

count    3866.000
mean        0.396
std         0.293
min         0.000
25%         0.159
50%         0.318
75%         0.589
max         1.000

more than 25% zero weeks: 2323 territories (60.1%)
more than 50% zero weeks: 1193 territories (30.9%)
more than 70% zero weeks:  741 territories (19.2%)
more than 80% zero weeks:  574 territories (14.8%)
more than 90% zero weeks:  410 territories (10.6%)
more than 95% zero weeks:  280 territories (7.2%)


Less of a problem than expected. The median territory has alerts in 73 of the 107 weeks, and only 15 per cent have more than 80 per cent zero weeks. One method with a separate rule for the sparse group should be enough.

In [8]:
active = (weekly["lost_per_1000ha"] > 0).groupby(weekly["territory_id"]).sum()
print("\nactive weeks per territory:")
print(active.describe().round(1).to_string())


active weeks per territory:
count    3866.0
mean       64.6
std        31.4
min         0.0
25%        44.0
50%        73.0
75%        90.0
max       107.0


The baseline is the 90th percentile of each territory's own history, rather than the 95th. With around 100 weeks of data, the 95th percentile depends on only a few observations and can move a lot because of one of them.

The week being evaluated is left out of the baseline. Otherwise, an extreme week could raise the threshold it has to beat and end up hiding itself.

In [9]:
BASELINE_PCT = 0.90
MIN_ACTIVE_WEEKS = 20

last_week = weekly["week"].max()

history = weekly[weekly["week"] < last_week]
current = weekly[weekly["week"] == last_week].set_index("territory_id")

stats = history.groupby("territory_id")["lost_per_1000ha"].agg(
    baseline=lambda s: s.quantile(BASELINE_PCT),
    median="median",
    active_weeks=lambda s: (s > 0).sum(),
)

scored = current[["lost_per_1000ha", "area_ha"]].join(stats)


print(f"{(scored['active_weeks'] < MIN_ACTIVE_WEEKS).sum()} territories below the history threshold")

547 territories below the history threshold


In [10]:
scored["ratio"] = scored["lost_per_1000ha"] / scored["baseline"]
scored["ratio"] = scored["ratio"].replace([float("inf"), -float("inf")], pd.NA).astype("Float64")

print(f"territories: {len(scored)}")
print(f"  baseline is zero (never lost enough to build one): {(scored['baseline'] == 0).sum()}")
print(f"  below {MIN_ACTIVE_WEEKS} active weeks: {(scored['active_weeks'] < MIN_ACTIVE_WEEKS).sum()}")
print()

usable = scored[(scored["baseline"] > 0) & (scored["active_weeks"] >= MIN_ACTIVE_WEEKS)]
print(f"usable for the percentile method: {len(usable)}")
print(usable["ratio"].describe().round(2).to_string())
print()
for t in [1, 1.5, 2, 3, 5]:
    print(f"ratio above {t}: {(usable['ratio'] > t).sum():>4} territories")

territories: 3866
  baseline is zero (never lost enough to build one): 410
  below 20 active weeks: 547

usable for the percentile method: 3319
count    3319.0
mean       0.23
std        1.93
min         0.0
25%         0.0
50%         0.0
75%        0.05
max       78.13

ratio above 1:  112 territories
ratio above 1.5:   72 territories
ratio above 2:   52 territories
ratio above 3:   37 territories
ratio above 5:   24 territories


A territory with a 90th percentile of 0.01 can end up with a huge ratio from a very small loss. So the same minimum loss threshold used earlier applies here too: the territory has to be outside its usual pattern and the loss has to be meaningful.

In [11]:
top = usable.nlargest(20, "ratio").join(
    territories.set_index("territory_id")[["name", "country"]]
)
print(top[["name", "country", "area_ha", "lost_per_1000ha", "baseline", "ratio", "active_weeks"]].round(3).to_string())

                                                                   name   country   area_ha  lost_per_1000ha  baseline   ratio  active_weeks
territory_id                                                                                                                                
3941                                                  Médio Rio Negro I    Brasil  1511.258            0.836     0.011  78.128           106
45160                                       Santa Ursula - Ampliación I      Perú     4.827            0.409     0.011  35.727            49
41357                            Nueva Esperanza del Mirim - Río Yavarí      Perú   141.016            0.763     0.027  28.265            75
30656                                                            Ñeñeña   Ecuador     0.677            4.588     0.167  27.498            32
13118                                                           Koriabo    Guyana    18.885            1.544     0.058  26.461            80
20457        

In [12]:
MIN_RATIO = 2

flagged = usable[
    (usable["ratio"] >= MIN_RATIO) & (usable["area_ha"] >= MIN_LOST_HA)
].copy()

print(f"{len(flagged)} territories flagged")
print()
out = flagged.nlargest(20, "ratio").join(territories.set_index("territory_id")[["name", "country"]])
print(out[["name", "country", "area_ha", "lost_per_1000ha", "baseline", "ratio"]].round(3).to_string())

23 territories flagged

                                                                      name   country   area_ha  lost_per_1000ha  baseline   ratio
territory_id                                                                                                                     
3941                                                     Médio Rio Negro I    Brasil  1511.258            0.836     0.011  78.128
41357                               Nueva Esperanza del Mirim - Río Yavarí      Perú   141.016            0.763     0.027  28.265
13118                                                              Koriabo    Guyana    18.885            1.544     0.058  26.461
20457            Resguardo Indígena Guahibo del Rio Siare o Barranco Lindo  Colombia    32.752            0.704     0.032  22.115
41366                                                Matses - Ampliación I      Perú    79.523            1.357     0.071  19.123
13119                                                             

Deforestation in the Amazon tends to increase during the dry season. If that seasonal effect is real, a flat baseline would overstate anomalies in August and understate them in March.

The median matters more than the mean here. The mean is heavily influenced by the 2024 fire season, so if the pattern only appears in that period, it is more likely to be one event than a recurring seasonal effect.

In [13]:
weekly["woy"] = weekly["week"].dt.isocalendar().week.astype(int)

seasonal = (
    weekly.groupby("woy")["lost_per_1000ha"]
    .agg(median="median", mean="mean", n="size")
    .round(4)
)
print(seasonal.to_string())

     median    mean      n
woy                       
1    0.0061  0.0470   7732
2    0.0016  0.0307   7732
3    0.0031  0.0462   7732
4    0.0022  0.0418   7732
5    0.0000  0.0339   7732
6    0.0025  0.0374   7732
7    0.0021  0.0478   7732
8    0.0000  0.0316   7732
9    0.0021  0.0435   7732
10   0.0000  0.0297   7732
11   0.0000  0.0288   7732
12   0.0029  0.0638   7732
13   0.0013  0.0408   7732
14   0.0000  0.0422   7732
15   0.0008  0.0420   7732
16   0.0021  0.0515   7732
17   0.0036  0.0571   7732
18   0.0009  0.0330   7732
19   0.0031  0.0766   7732
20   0.0017  0.0451   7732
21   0.0009  0.0442   7732
22   0.0003  0.0564   7732
23   0.0043  0.0614   7732
24   0.0026  0.0582   7732
25   0.0069  0.0698   7732
26   0.0059  0.0684   7732
27   0.0106  0.1016   7732
28   0.0086  0.1353   7732
29   0.0144  0.1105   7732
30   0.0140  0.1289   7732
31   0.0087  0.1062  11598
32   0.0128  0.1676  11598
33   0.0061  0.1551  11598
34   0.0535  0.2932   7732
35   0.0417  0.2493   7732
3

It shows up in the median, so the seasonal effect is real. Weeks 1 to 22 range from 0.000 to 0.0061, while weeks 34 to 43 range from 0.014 to 0.0535. That's roughly a 10x difference between the wet and dry seasons.

Two years of history only give us two observations for each week of the year, which isn't enough to estimate a percentile. Using a four-week window on either side gives us around twenty observations instead.

Week 52 and week 1 are next to each other, not 50 weeks apart, so the distance is calculated circularly.

In [14]:
SEASON_WINDOW = 4

history = weekly[weekly["week"] < last_week]
current = weekly[weekly["week"] == last_week].set_index("territory_id")

target_woy = int(weekly.loc[weekly["week"] == last_week, "woy"].iloc[0])
print(f"evaluating week {target_woy}")

d = (history["woy"] - target_woy).abs()
d = d.where(d <= 26, 52 - d)
window = history[d <= SEASON_WINDOW]

seas = window.groupby("territory_id")["lost_per_1000ha"].agg(
    seasonal_baseline=lambda s: s.quantile(BASELINE_PCT),
    window_obs="size",
)

print(seas["window_obs"].describe().round(1).to_string())

evaluating week 33
count    3866.0
mean       20.0
std         0.0
min        20.0
25%        20.0
50%        20.0
75%        20.0
max        20.0


## What the seasonal adjustment changes

In [15]:
scored_s = scored.join(seas)
scored_s["ratio_seasonal"] = scored_s["lost_per_1000ha"] / scored_s["seasonal_baseline"]
scored_s["ratio_seasonal"] = scored_s["ratio_seasonal"].replace([float("inf"), -float("inf")], pd.NA).astype("Float64")

print("baseline comparison")
print(f"  flat baseline, median:     {scored_s['baseline'].median():.4f}")
print(f"  seasonal baseline, median: {scored_s['seasonal_baseline'].median():.4f}")
print()

both = scored_s[(scored_s["baseline"] > 0) & (scored_s["seasonal_baseline"] > 0) & (scored_s["area_ha"] >= 5)]
print(f"territories with real loss and both baselines: {len(both)}")
print(f"  flagged by flat ratio >= 2:     {(both['ratio'] >= 2).sum()}")
print(f"  flagged by seasonal ratio >= 2: {(both['ratio_seasonal'] >= 2).sum()}")

baseline comparison
  flat baseline, median:     0.1024
  seasonal baseline, median: 0.1600

territories with real loss and both baselines: 95
  flagged by flat ratio >= 2:     23
  flagged by seasonal ratio >= 2: 19


In [16]:
dropped = both[(both["ratio"] >= 2) & (both["ratio_seasonal"] < 2)].join(
    territories.set_index("territory_id")[["name", "country"]]
)
print("dropped by seasonal adjustment:")
print(dropped[["name", "country", "area_ha", "baseline", "seasonal_baseline", "ratio", "ratio_seasonal"]].round(3).to_string())

added = both[(both["ratio"] < 2) & (both["ratio_seasonal"] >= 2)].join(
    territories.set_index("territory_id")[["name", "country"]]
)
print(f"\nadded by seasonal adjustment: {len(added)}")
if len(added):
    print(added[["name", "country", "area_ha", "ratio", "ratio_seasonal"]].round(3).to_string())

dropped by seasonal adjustment:
                           name  country  area_ha  baseline  seasonal_baseline   ratio  ratio_seasonal
territory_id                                                                                          
3628          Cacau do Tarauacá   Brasil    5.923     0.081              0.108    2.54             1.9
13052                   Akawini   Guyana   27.529     0.040              2.616  13.497           0.206
30312                   Sehuaya  Ecuador    9.663     0.193              0.369   2.133           1.118
40620            Nuevo Mayoruna     Perú    5.009     0.717              1.559   2.565            1.18
42337               San Antonio     Perú    5.442     1.082              2.661   2.496           1.015

added by seasonal adjustment: 1
                                                                                                    name   country  area_ha  ratio  ratio_seasonal
territory_id                                                       

Akawini is the clearest example. Its flat baseline is 0.040, while its seasonal baseline is 2.616, about 65 times higher. It loses a lot during the dry season every year. With the flat baseline, it ranked fourth with a ratio of 13.5. After the seasonal adjustment, the ratio drops to 0.21, meaning this August it was only at about a fifth of what it normally loses.

Without the adjustment, Akawini would have been flagged for what is actually a normal August. The opposite happens too. RI Guahibo y Cuiva goes from 1.0 to 3.2: roughly its usual annual loss, but unusually high for that time of year.

So the adjustment isn't just removing noise. It changes which territories look unusual, in both directions.

In [17]:
final = scored_s[
    (scored_s["area_ha"] >= MIN_LOST_HA)
    & (scored_s["ratio_seasonal"] >= MIN_RATIO)
].copy()

final["score"] = final["ratio_seasonal"] * (final["area_ha"] ** 0.5)

out = final.nlargest(20, "score").join(territories.set_index("territory_id")[["name", "country"]])
print(f"{len(final)} territories flagged\n")
print(out[["name", "country", "area_ha", "lost_per_1000ha", "seasonal_baseline", "ratio_seasonal", "score"]].round(2).to_string())

19 territories flagged

                                                                                                    name   country  area_ha  lost_per_1000ha  seasonal_baseline  ratio_seasonal    score
territory_id                                                                                                                                                                            
3941                                                                                   Médio Rio Negro I    Brasil  1511.26             0.84               0.01           72.29  2810.28
41357                                                             Nueva Esperanza del Mirim - Río Yavarí      Perú   141.02             0.76               0.01           63.77   757.31
20457                                          Resguardo Indígena Guahibo del Rio Siare o Barranco Lindo  Colombia    32.75             0.70               0.01           64.66   370.06
4068                                               

548 territories either have a seasonal baseline of zero or too few active weeks for a percentile to be useful. There is no meaningful baseline to compare against, so for these territories the absolute loss threshold is the only criterion. If a territory almost never loses forest, a material loss is already unusual.

In [18]:
sparse = scored_s[
    (scored_s["seasonal_baseline"].isna())
    | (scored_s["seasonal_baseline"] == 0)
    | (scored_s["active_weeks"] < MIN_ACTIVE_WEEKS)
].copy()

print(f"{len(sparse)} territories without a usable baseline")
print(f"  lost nothing this week:   {(sparse['area_ha'] == 0).sum()}")
print(f"  lost something:           {(sparse['area_ha'] > 0).sum()}")
print(f"  lost 5 ha or more:        {(sparse['area_ha'] >= MIN_LOST_HA).sum()}")
print()
print(sparse.loc[sparse['area_ha'] > 0, "area_ha"].describe().round(2).to_string())

548 territories without a usable baseline
  lost nothing this week:   543
  lost something:           5
  lost 5 ha or more:        0

count    5.00
mean     0.39
std      0.78
min      0.01
25%      0.01
50%      0.02
75%      0.10
max      1.78


The ratio tells us how unusual the loss is, while absolute area tells us how much it matters. Ranking by ratio alone pushes small territories with large relative spikes above events of a thousand hectares.

The score combines both. The square root of the area keeps large events important without letting one huge event completely dominate the ranking.

In [19]:
scored_s["method"] = "seasonal_baseline"
scored_s.loc[
    scored_s["seasonal_baseline"].isna()
    | (scored_s["seasonal_baseline"] == 0)
    | (scored_s["active_weeks"] < MIN_ACTIVE_WEEKS),
    "method",
] = "sparse_history"

flagged = scored_s[
    (scored_s["area_ha"] >= MIN_LOST_HA)
    & (
        (scored_s["ratio_seasonal"] >= MIN_RATIO)
        | (scored_s["method"] == "sparse_history")
    )
].copy()

flagged["score"] = flagged["ratio_seasonal"].fillna(MIN_RATIO) * (flagged["area_ha"] ** 0.5)

print(flagged["method"].value_counts().to_string())
print(f"\ntotal flagged: {len(flagged)}")

method
seasonal_baseline    19

total flagged: 19


## Conclusions

**The criterion.** A territory is flagged when it loses at least 5 ha in a week and that loss is at least twice its seasonal baseline, defined as the 90th percentile of its own history for that time of year. Nineteen of the 3,866 territories meet both conditions in the week evaluated.

**Seasonality is real and had to be accounted for.** Median weekly loss varies by roughly a factor of ten between the wet and dry seasons. A flat baseline overstates anomalies in August. The seasonal adjustment removed five of the 23 territories flagged by the flat baseline and added one that it missed.

**Two thresholds, not one.** Being outside the usual pattern and losing a meaningful amount are different things. Using either one on its own produces a list that isn't very useful.

**The method catches what a relative ranking misses.** Médio Rio Negro I lost 1,511 ha in one week, 72 times its seasonal baseline. It does not show up when ranking loss relative to territory size because it is a large territory. Vale do Javari and Alto Rio Negro have the same issue.

### Known limitations

**The 2024 fire season affects the baseline.** Seventy per cent of all loss in the window falls in three months of 2024 and is mostly fire, not clearing. Using these two years as the baseline makes that fire season look like normal October behaviour. Fixing this would mean either using more years of history or using GFW's alert-driver classification to separate the causes.

**Two years is not much for a seasonal model.** Each week of the year only has two observations, expanded to around twenty with the four-week window. That's enough to work with, but not enough to be very confident about any individual week.

**The thresholds are choices, not findings.** The 90th percentile, 5 ha minimum and ratio of 2 are documented parameters, chosen based on what a small team could realistically act on. They are kept as constants at the top of the script for that reason.

Implemented in `src/detect_anomalies.py`.